# Phase 5 — Machine Learning Models

Thin wrapper around `src/train_ml.py` (Random Forest + XGBoost, rolling-origin CV, SHAP, serialized models). Run the script directly for the full loop over all three modes:

```bash
python -m src.train_ml
```

This notebook re-runs it for ONE mode so you can inspect the CV search and SHAP plot inline instead of just reading console output.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from src import config, evaluate
from src.features import time_based_split
from src.train_ml import get_feature_columns, tune_and_fit, save_shap_plot, RF_PARAM_GRID, XGB_PARAM_GRID
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
MODE = 'total'  # change to 'bus' or 'rail' to inspect the others

features_df = pd.read_csv(config.FEATURES_CSV_PATH, parse_dates=[config.DATE_COL])
mode_df = features_df[features_df[config.MODE_COL] == MODE].sort_values(config.DATE_COL)
train_df, val_df, test_df = time_based_split(mode_df)
trainval_df = pd.concat([train_df, val_df]).sort_values(config.DATE_COL)

feature_cols = get_feature_columns(mode_df)
X_trainval, y_trainval = trainval_df[feature_cols], trainval_df[config.TARGET_COL]
X_test, y_test = test_df[feature_cols], test_df[config.TARGET_COL]
X_trainval.shape, X_test.shape

## Random Forest — rolling-origin CV

In [ ]:
rf_best, rf_params = tune_and_fit(
    RandomForestRegressor(random_state=config.RANDOM_SEED), RF_PARAM_GRID,
    X_trainval, y_trainval,
)
rf_params

In [ ]:
rf_pred = rf_best.predict(X_test)
evaluate.score_all(y_test, rf_pred)

## XGBoost — rolling-origin CV

In [ ]:
xgb_best, xgb_params = tune_and_fit(
    XGBRegressor(random_state=config.RANDOM_SEED, objective='reg:squarederror', tree_method='hist'),
    XGB_PARAM_GRID, X_trainval, y_trainval,
)
xgb_params

In [ ]:
xgb_pred = xgb_best.predict(X_test)
evaluate.score_all(y_test, xgb_pred)

## SHAP feature importance (XGBoost)

In [ ]:
import matplotlib.pyplot as plt
import shap

explainer = shap.TreeExplainer(xgb_best)
sample = X_test.sample(min(len(X_test), 300), random_state=config.RANDOM_SEED)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, plot_type='bar')

## Plot actual vs. predicted on the test window

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(test_df[config.DATE_COL], y_test, label='Actual')
plt.plot(test_df[config.DATE_COL], xgb_pred, label='XGBoost predicted')
plt.legend(); plt.title(f'{MODE} — test window'); plt.xticks(rotation=45); plt.tight_layout()
plt.show()

To log these to `reports/ml_metrics.csv` and serialize the models the way the app expects, run the full script instead of this notebook: `python -m src.train_ml`.